<a href="https://colab.research.google.com/github/CaroleSchoepfer5/BugNet_SoilArthro/blob/main/BugNet_SoilArthro_FlatBug.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install rawpy tqdm pillow

# Clone the flat-bug repo
!git clone https://github.com/darsa-group/flat-bug.git --branch main --single-branch flat-bug

# --- Small compatibility patch (because Colab often uses Python 3.10) ---

import re
import os

def find_and_replace_in_file(file_path, search_pattern, replacement_text):
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        updated_content = re.sub(search_pattern, replacement_text, content)
        with open(file_path, 'w', encoding='utf-8') as file:
            file.write(updated_content)
        print(f"Replaced text in '{file_path}' successfully.")
    except FileNotFoundError:
        print(f"File '{file_path}' not found.")
    except Exception as e:
        print(f"An error occurred: {e}")

# Allow Python 3.10
find_and_replace_in_file(
    'flat-bug/pyproject.toml',
    r'requires-python = ">=3.11"',
    'requires-python = ">=3.10"'
)

# Add typing_extensions.Self where needed
self_replace = "from typing_extensions import Self"
find_and_replace_in_file(
    'flat-bug/src/flat_bug/predictor.py',
    r'from typing import Any, List, Optional, Self, Tuple, Union',
    f'from typing import Any, List, Optional, Tuple, Union\n{self_replace}'
)
find_and_replace_in_file(
    'flat-bug/src/flat_bug/augmentations.py',
    r'from typing import Dict, List, Optional, Self, Tuple, Union',
    f'from typing import Dict, List, Optional, Tuple, Union\n{self_replace}'
)
find_and_replace_in_file(
    'flat-bug/src/flat_bug/trainers.py',
    r'from typing import Any, Dict, List, Optional, Self, Tuple, Union',
    f'from typing import Any, Dict, List, Optional, Tuple, Union\n{self_replace}'
)
find_and_replace_in_file(
    'flat-bug/src/flat_bug/datasets.py',
    r'from typing import Dict, List, Optional, Self, Tuple, Union',
    f'from typing import Dict, List, Optional, Tuple, Union\n{self_replace}'
)

# Install flat-bug package in editable mode
!pip install -e flat-bug

# Make sure Python can find it
import sys
sys.path.append("/content/flat-bug/src")

print("Setup finished.")


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 44.9 MB/s eta 0:00:00
Cloning into 'flat-bug'...
remote: Enumerating objects: 3217, done.
remote: Counting objects: 100% (292/292), done.
remote: Compressing objects: 100% (206/206), done.
remote: Total 3217 (delta 115), reused 102 (delta 86), pack-reused 2925 (from 2)
Receiving objects: 100% (3217/3217), 45.36 MiB | 22.83 MiB/s, done.
Resolving deltas: 100% (1856/1856), done.
Replaced text in 'flat-bug/pyproject.toml' successfully.
Replaced text in 'flat-bug/src/flat_bug/predictor.py' successfully.
Replaced text in 'flat-bug/src/flat_bug/augmentations.py' successfully.
Replaced text in 'flat-bug/src/flat_bug/trainers.py' successfully.
Replaced text in 'flat-bug/src/flat_bug/datasets.py' successfully.
Obtaining file:///content/flat-bug
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable 

In [ ]:
import os, glob, json, io, uuid, re, shutil, tempfile
from typing import List, Tuple, Union, Optional

from tqdm import tqdm
import numpy as np
import torch
import rawpy
from PIL import Image

from flat_bug.predictor import Predictor, TensorPredictions

# ---------- Helper functions ----------

def parse_image(
    images : Optional[
        Union[
            np.ndarray, bytes, str,
            Union[List[Union[np.ndarray, bytes, str]], Tuple[Union[np.ndarray, bytes, str]]]
        ]
    ],
    device : Union[torch.device, str] = "cpu"
):
    # List/Tuple: recursively parse each image
    if isinstance(images, (list, tuple)):
        return [parse_image(image, device) for image in images]
    # String: path to image
    elif isinstance(images, str):
        if re.search(re.compile(r"\.dng$", re.IGNORECASE), images):
            with rawpy.imread(images) as raw:
                images = raw.postprocess()
                images = Image.fromarray(images)
        else:
            images = Image.open(images)
    # Bytes: open from memory
    elif isinstance(images, bytes):
        images = Image.open(io.BytesIO(images))
    # Numpy array: use directly
    elif isinstance(images, np.ndarray):
        pass
    else:
        raise ValueError(
            f"Expected image(s) to be a np.ndarray, string or bytes, "
            f"or list of these, but got {type(images)}"
        )

    # Convert to torch tensor (C, H, W)
    image = np.array(images)
    return torch.from_numpy(image).permute(2, 0, 1).to(device)

def generate_uuid() -> str:
    # Just a short unique ID
    return str(uuid.uuid4())[::3]

import time

def wait_until_crops_finished(folder, identifier, timeout=60, stable_time=1.2):
    """
    Wait until the number of crop files stops changing.
    This is needed because FlatBug saves crops asynchronously.
    """
    pattern = os.path.join(folder, f"crop*{identifier}.png")

    start = time.time()
    last_count = -1
    last_change = time.time()

    while time.time() - start < timeout:
        files = glob.glob(pattern)
        count = len(files)

        if count != last_count:
            last_count = count
            last_change = time.time()

        if time.time() - last_change >= stable_time:
            return sorted(files)

        time.sleep(0.2)

    return sorted(glob.glob(pattern))


# ---------- Localizer model wrapper ----------

class Localizer(Predictor):
    def predict(
        self,
        images : Optional[
            Union[
                np.ndarray, bytes, str,
                Union[List[Union[np.ndarray, bytes, str]], Tuple[Union[np.ndarray, bytes, str]]]
            ]
        ],
        do_plot : bool | List[bool] = False,
        include_crops : bool = False,
        outdir : str = "output"
    ) -> dict:
        """
        Run the model on one or many images and save crops & (optional) plots.

        Returns a dict with:
          - 'uuids'
          - 'predictions'
          - 'crops'  (list of lists of crop file paths)
          - 'visualizations'
        """
        data = {
            "uuids": [],
            "predictions": [],
            "crops" : [],
            "visualizations": []
        }

        if not isinstance(images, (list, tuple)):
            images = [images]

        if not isinstance(do_plot, list):
            if isinstance(do_plot, tuple):
                do_plot = list(do_plot)
            else:
                do_plot = [do_plot]
            if len(do_plot) == 1 and len(images) > 1:
                do_plot = do_plot * len(images)

        if not all([isinstance(plot, bool) for plot in do_plot]):
            raise ValueError(f"Expected do_plot to be a boolean or list of booleans, but got {do_plot}")

        for i, image in enumerate(tqdm(images, desc="Localizing insects", unit="image", leave=True)):
            if isinstance(image, str):
                image_identifier = os.path.splitext(os.path.basename(image))[0]
            else:
                image_identifier = "DUMMY"

            # Load the image
            image_tensor = parse_image(image, self._device)

            # Unique ID for this image’s outputs
            identifier = generate_uuid()

            # Output directory for this image (inside outdir)
            this_outdir = os.path.join(outdir, identifier)
            if not os.path.exists(this_outdir):
                os.makedirs(this_outdir)

            # Run model
            predictions : TensorPredictions = self.pyramid_predictions(
                image_tensor, "DUMMY_PATH_STR", scale_before=1
            )

            # Save crops (filenames will include image_identifier and identifier)
            predictions.save_crops(
                outdir=this_outdir,
                basename=image_identifier,
                mask=True,
                identifier=identifier
            )

            # Wait until all async crop-saving tasks for this image are finished
            crops = wait_until_crops_finished(this_outdir, identifier)

            print(
                image_identifier,
                "detections:",
                len(predictions.json_data),
               "saved:",
                len(glob.glob(os.path.join(this_outdir,
                                          f"crop*{identifier}.png")))
            )

            # Optionally create visualizations (we don’t really need them for batch)
            if do_plot[i]:
                visualization_dir = os.path.join(os.path.dirname(this_outdir), "visualization")
                if not os.path.exists(visualization_dir):
                    os.makedirs(visualization_dir)
                predict_image = os.path.join(visualization_dir, f'{identifier}_visualization.jpg')
                predictions.plot(outpath=predict_image, scale=1/2)
            else:
                predict_image = None

            data["uuids"].append(identifier)
            data["visualizations"].append(predict_image)
            data["crops"].append(crops)
            data["predictions"].append(predictions.json_data)

        return data


def get_defaults():
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    dtype = torch.float16
    return device, dtype


# ---------- Batch function you will call ----------

def batch_flatbug(
    input_folder: str,
    output_folder: str,
    weights_path: str,
    score_threshold: float = 0.25,
):
    """
    Run FlatBug on all images in a folder, save all crops into one output folder
    with names: OriginalName_1.png, OriginalName_2.png, ...

    input_folder: folder with your JPG/PNG/DNG images
    output_folder: where all the crops (and JSON) will go
    weights_path: path to your .pt model file
    """

    # 1) Collect images recursively
    image_paths = []
    for root, dirs, files in os.walk(input_folder):
        for f in files:
           if f.lower().endswith((".jpg", ".jpeg", ".png", ".dng")):
              image_paths.append(os.path.join(root, f))

    image_paths = sorted(image_paths)


    if not image_paths:
        raise ValueError(f"No images found in {input_folder}")

    print(f"Found {len(image_paths)} images.")

    # 2) Prepare output folders
    os.makedirs(output_folder, exist_ok=True)
    tmp_outdir = os.path.join(output_folder, "_tmp_flatbug")
    os.makedirs(tmp_outdir, exist_ok=True)

    # 3) Set up device, dtype and model
    device, dtype = get_defaults()
    print(f"Using device: {device}, dtype: {dtype}")

    model = Localizer(model=weights_path, device=device, dtype=dtype)
    model.set_hyperparameters(
        SCORE_THRESHOLD=score_threshold,
        EDGE_CASE_MARGIN=32,
        MIN_MAX_OBJ_SIZE=(16, 768),
        TIME=False,
    )

    # 4) Run predictions
    results = model.predict(
        images=image_paths,
        do_plot=False,
        include_crops=False,
        outdir=tmp_outdir,
    )

    # 5) Move & rename crops
    total_crops = 0

    for img_path, crop_paths in zip(image_paths, results["crops"]):
        base = os.path.splitext(os.path.basename(img_path))[0]

        for idx, crop_path in enumerate(crop_paths, start=1):
            if not os.path.isfile(crop_path):
                continue

            new_name = f"{base}_{idx}.png"
            new_path = os.path.join(output_folder, new_name)

            shutil.move(crop_path, new_path)
            total_crops += 1

    # # 6) Save per-image JSON (optional but often useful)
    # for img_path, pred, uuid_ in zip(
    #     image_paths,
    #     results["predictions"],
    #     results["uuids"],
    # ):
    #     base = os.path.splitext(os.path.basename(img_path))[0]
    #     json_name = f"{base}_{uuid_}.json"
    #     json_path = os.path.join(output_folder, json_name)
    #     with open(json_path, "w", encoding="utf-8") as f:
    #         json.dump(pred, f)

    # 7) Clean up temporary directory
    shutil.rmtree(tmp_outdir, ignore_errors=True)

    #tqdm.write(
    #    f"{image_identifier}: detections={n_det}, saved={len(crops)}"
    #)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# >>> EDIT THESE THREE LINES TO MATCH YOUR DRIVE <<<

input_folder = "/content/drive/MyDrive/FlatBug/input"   # where you put your test images
output_folder = "/content/drive/MyDrive/FlatBug/output" # where you want crops to go
weights_path = "/content/drive/MyDrive/FlatBug/models/flatbug_weights.pt"  # your .pt file

batch_flatbug(
    input_folder=input_folder,
    output_folder=output_folder,
    weights_path=weights_path,
    score_threshold=0.22,   # you can tweak this later
)


Found 171 images.
Using device: cuda:0, dtype: torch.float16
YOLOv8m-seg summary (fused): 105 layers, 27,222,963 parameters, 0 gradients, 104.3 GFLOPs


Localizing insects:   1%|          | 1/171 [00:08<24:39,  8.70s/image]

Bruech02A_3 (1) detections: 12 saved: 23


Localizing insects:   1%|          | 2/171 [00:13<17:48,  6.32s/image]

Bruech06A_3 (1) detections: 12 saved: 7


...

Bruech23A_3 (7) detections: 12 saved: 3


Localizing insects:  99%|█████████▉| 170/171 [14:30<00:05,  5.03s/image]

Bruech23A_3 (8) detections: 12 saved: 6


Localizing insects: 100%|██████████| 171/171 [14:34<00:00,  5.12s/image]

Bruech23A_3 (9) detections: 12 saved: 2
